# 👗 VirtualFit AI – Genuine LEFFA (CVPR 2025) Google Colab Runner

Welcome to the official **VirtualFit AI** GPU runner powered by **LEFFA** (*Learning Flow Fields in Attention for Controllable Person Image Generation*, CVPR 2025).

### ⚡ Features & Optimizations in this Notebook:
- **Zero-Crash Float16 Streamer**: Direct `mmap=True` streaming and in-place `float16` weight casting cuts CPU RAM to <4.5 GB.
- **Targeted ~8 GB Download**: Omits SDXL (7 GB) and Pose-Transfer (5 GB), preventing disk saturation.
- **Stage-by-Stage VRAM Eviction**: DensePose and Human Parsing models are evicted from GPU before diffusion starts.
- **Self-Healing & Repeatable**: Re-run anytime or run multiple outfits in the same session without re-downloading!

---

### 🚀 Quick Setup Instructions:
1. **Ensure Free T4 GPU is Active**: In the menu above, click `Runtime` ➔ `Change runtime type` ➔ Select `T4 GPU` ➔ Click `Save`.
2. Run **Step 1: Check System Memory & NVIDIA GPU**.
3. Run **Step 2: Install Dependencies & Setup LEFFA Repository**.
4. Run **Step 3: Download Targeted Checkpoints** (~8 GB, takes ~2 mins).
5. Run **Step 4: Upload VirtualFit ZIP** (`virtualfit_input_<taskId>.zip`).
6. Run **Step 5: Execute Memory-Optimized LEFFA Virtual Try-On**.
7. Run **Step 6: Preview Results & Download Result ZIP**.

> **Tip for Subsequent Outfits**: If you want to try on another garment in the same session, you **only** need to run **Step 4, Step 5, and Step 6**!

In [ ]:
# Step 1: Check System Memory & NVIDIA GPU Runtime
import os
import sys
import gc
import torch

def get_system_ram():
    try:
        import psutil
        mem = psutil.virtual_memory()
        return mem.total / (1024**3), mem.available / (1024**3), mem.used / (1024**3)
    except Exception:
        return 12.7, 10.0, 2.7

total_ram, avail_ram, used_ram = get_system_ram()
print("=" * 65)
print("VIRTUALFIT AI – HARDWARE & MEMORY PRE-FLIGHT AUDIT")
print("=" * 65)
print(f"[*] Host System RAM:  Total: {total_ram:.2f} GB | Available: {avail_ram:.2f} GB | Used: {used_ram:.2f} GB")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free_vram, total_vram_alt = torch.cuda.mem_get_info()
    free_vram_gb = free_vram / (1024**3)
    print(f"[*] GPU Model:        {gpu_name}")
    print(f"[*] Total GPU VRAM:   {total_vram:.2f} GB")
    print(f"[*] Free GPU VRAM:    {free_vram_gb:.2f} GB")
    print("✅ GPU Memory is sufficient for optimized float16 LEFFA inference!")
    !nvidia-smi
else:
    print("\n" + "!" * 65)
    print("❌ ERROR: NO GPU DETECTED IN THIS COLAB SESSION!")
    print("   Your Colab runtime is currently set to CPU.")
    print("   LEFFA requires an NVIDIA GPU (e.g. Free T4 GPU) to run diffusion.")
    print("\n👉 ACTION REQUIRED (Takes 5 seconds):")
    print("   1. Click 'Runtime' in the top Google Colab menu.")
    print("   2. Click 'Change runtime type'.")
    print("   3. Under 'Hardware accelerator', choose 'T4 GPU'.")
    print("   4. Click 'Save'.")
    print("   5. Re-run Step 1!")
    print("!" * 65 + "\n")
    raise RuntimeError("Colab runtime is set to CPU. Please switch to 'T4 GPU' under Runtime -> Change runtime type.")


In [ ]:
# Step 2: Install Dependencies & Setup Official LEFFA Repository
import os
import sys
import torch

# Verify CUDA status
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available:  {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("Colab runtime is set to CPU. Please switch to 'T4 GPU' in Runtime -> Change runtime type.")

print("[*] 1/3 Installing Python dependencies (diffusers, transformers, av, onnxruntime-gpu)...")
!pip install -q av diffusers transformers accelerate fvcore timm omegaconf einops scikit-image huggingface_hub onnxruntime-gpu opencv-python tqdm pillow psutil pycocotools pyyaml cloudpickle scipy

print("[*] 2/3 Ensuring official LEFFA repository is cloned and verified...")
LEFFA_ROOT = "/content/Leffa"

if not os.path.exists(os.path.join(LEFFA_ROOT, "leffa", "transform.py")):
    print("    Cloning fresh repository from https://github.com/franciszzj/Leffa.git...")
    !rm -rf /content/Leffa
    !git clone https://github.com/franciszzj/Leffa.git /content/Leffa
else:
    print("    LEFFA repository verified at /content/Leffa")

print("[*] 3/3 Configuring Python import search paths & writing verified inference pipeline...")
for p in [LEFFA_ROOT, os.path.join(LEFFA_ROOT, "3rdparty")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# Clean write of official inference.py with LeffaPipeline and safe generator fallback
inference_py = os.path.join(LEFFA_ROOT, "leffa", "inference.py")
if os.path.exists(os.path.dirname(inference_py)):
    with open(inference_py, "w", encoding="utf-8") as f:
        f.write("""from typing import Any, Dict, Optional
import numpy as np
import torch
import torch.nn as nn
from leffa.pipeline import LeffaPipeline

class LeffaInference(object):
    def __init__(self, model: nn.Module) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device)
        self.model.eval()
        self.pipe = LeffaPipeline(model=self.model, device=self.device)

    def to_gpu(self, data: Dict[str, Any]) -> Dict[str, Any]:
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                data[k] = v.to(self.device)
        return data

    def __call__(self, data: Dict[str, Any], **kwargs) -> Dict[str, Any]:
        data = self.to_gpu(data)
        ref_acceleration = kwargs.get("ref_acceleration", False)
        num_inference_steps = kwargs.get("num_inference_steps", 50)
        guidance_scale = kwargs.get("guidance_scale", 2.5)
        seed = kwargs.get("seed", 42)
        repaint = kwargs.get("repaint", False)
        try:
            generator = torch.Generator(self.pipe.device).manual_seed(seed)
        except Exception:
            generator = torch.Generator("cpu").manual_seed(seed)
        images = self.pipe(
            src_image=data["src_image"],
            ref_image=data["ref_image"],
            mask=data["mask"],
            densepose=data["densepose"],
            ref_acceleration=ref_acceleration,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
            repaint=repaint,
        )[0]
        outputs = {}
        outputs["src_image"] = (data["src_image"] + 1.0) / 2.0
        outputs["ref_image"] = (data["ref_image"] + 1.0) / 2.0
        outputs["generated_image"] = images
        return outputs
""")

print("✅ LEFFA environment successfully configured and verified!")


In [ ]:
# Step 3: Targeted Checkpoint Download (Selective, ~8 GB instead of 25 GB)
import os
from huggingface_hub import snapshot_download

CKPTS_DIR = "/content/ckpts"
os.makedirs(CKPTS_DIR, exist_ok=True)

print("=" * 65)
print(f"[*] Downloading TARGETED LEFFA weights into {CKPTS_DIR}...")
print("    Selective filter: Omits SDXL (7 GB) & Pose Transfer (5 GB) to prevent RAM/Disk exhaustion.")
print("=" * 65)

snapshot_download(
    repo_id="franciszzj/Leffa",
    local_dir=CKPTS_DIR,
    allow_patterns=[
        "stable-diffusion-inpainting/**",
        "virtual_tryon.pth",
        "virtual_tryon_dc.pth",
        "densepose/**",
        "humanparsing/**",
        "openpose/**",
        "schp/**"
    ],
    ignore_patterns=[
        "stable-diffusion-xl-1.0-inpainting-0.1/**",
        "pose_transfer.pth",
        "examples/**"
    ]
)

vt_path = os.path.join(CKPTS_DIR, "virtual_tryon.pth")
densepose_path = os.path.join(CKPTS_DIR, "densepose", "model_final_162be9.pkl")

if os.path.exists(vt_path) and os.path.exists(densepose_path):
    print("\n" + "=" * 65)
    print("🎉 TARGETED LEFFA CHECKPOINTS VERIFIED ON DISK!")
    print(f"   • Virtual Try-On (Standard): {os.path.getsize(vt_path)/(1024**2):.1f} MB")
    print(f"   • DensePose Backbone:        {os.path.getsize(densepose_path)/(1024**2):.1f} MB")
    print("=" * 65)
else:
    raise FileNotFoundError("❌ Checkpoint download incomplete. Please re-run Step 3.")


In [ ]:
# Step 4: Upload VirtualFit Input ZIP & Configure Lazy Loading Parameters
import os
import glob
import shutil
import zipfile
import json
from google.colab import files

WORKSPACE_DIR = "/content/virtualfit_workspace"
INPUT_DIR = os.path.join(WORKSPACE_DIR, "input")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "output")

# Clear previous task input to ensure a completely fresh run
if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📥 Upload your 'virtualfit_input_<id>.zip' downloaded from VirtualFit AI Studio:")
uploaded = files.upload()
input_zip_filename = None

if uploaded:
    for fn in uploaded.keys():
        if fn.endswith(".zip"):
            input_zip_filename = fn
            break
else:
    existing_zips = sorted(
        glob.glob("/content/virtualfit_input*.zip") + glob.glob(os.path.join(WORKSPACE_DIR, "virtualfit_input*.zip")),
        key=os.path.getmtime
    )
    if existing_zips:
        input_zip_filename = existing_zips[-1]
        print(f"📦 Using most recent uploaded package: {input_zip_filename}")

if not input_zip_filename or not os.path.exists(input_zip_filename):
    raise ValueError("❌ No ZIP file was found or uploaded. Please re-run this cell and upload your ZIP.")

if not os.path.isabs(input_zip_filename):
    input_zip_filename = os.path.abspath(input_zip_filename)

print(f"[*] Extracting package: {input_zip_filename}...")
with zipfile.ZipFile(input_zip_filename, 'r') as zip_ref:
    for member in zip_ref.namelist():
        if ".." in member or member.startswith("/") or member.startswith("\\"):
            raise ValueError(f"Security Alert: Unsafe filename in ZIP: {member}")
    zip_ref.extractall(INPUT_DIR)

meta_path = os.path.join(INPUT_DIR, "metadata.json")
if not os.path.exists(meta_path):
    raise FileNotFoundError("Missing metadata.json in input ZIP package.")

with open(meta_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Map category to LEFFA vt_garment_type
raw_cat = metadata.get("category", "tops")
garment_type = metadata.get("leffa_garment_type")
if not garment_type:
    cat_map = {"tops": "upper_body", "bottoms": "lower_body", "one-pieces": "dresses"}
    garment_type = cat_map.get(raw_cat, "upper_body")

# Select ONLY the one required checkpoint (viton_hd for tops/bottoms, dress_code for dresses)
model_type = metadata.get("leffa_model_type", "dress_code" if garment_type == "dresses" else "viton_hd")
weights_filename = "virtual_tryon_dc.pth" if model_type == "dress_code" else "virtual_tryon.pth"

print("\n--- Verified VirtualFit Task Package ---")
print(f"Task ID:            {metadata.get('generation_id')}")
print(f"Canonical Category: {raw_cat}")
print(f"LEFFA Garment Type: {garment_type} (upper_body / lower_body / dresses)")
print(f"Active Checkpoint:  {weights_filename} (SINGLE MODEL LOAD)")
print(f"Person Model:       {metadata.get('model_name')}")
print(f"Garment Name:       {metadata.get('garment_name')}")
print("✅ Input assets successfully validated for single-model LEFFA pipeline!")


In [ ]:
# Step 5: Execute Memory-Optimized LEFFA Virtual Try-On (Self-Healing & Repeatable)
import os
import sys
import time
import gc
import json
import torch
import numpy as np
from PIL import Image

# 0. Immediate GPU Guard: LEFFA requires an NVIDIA GPU for diffusion inference
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "=" * 65 + "\n"
        "❌ NO GPU DETECTED IN THIS COLAB SESSION!\n"
        "   Notice: Your session is currently running on CPU.\n"
        "   LEFFA diffusion requires CUDA acceleration and CANNOT run on CPU.\n\n"
        "👉 HOW TO FIX (Takes 5 seconds):\n"
        "   1. Click 'Runtime' in the top Google Colab menu bar.\n"
        "   2. Click 'Change runtime type'.\n"
        "   3. Under 'Hardware accelerator', choose 'T4 GPU'.\n"
        "   4. Click 'Save'.\n"
        "   5. Re-run Step 1 through Step 5!\n"
        + "=" * 65
    )

# 1. Dependency Auto-Healer: Automatically installs missing packages if session was restarted
required_packages = ["onnxruntime", "diffusers", "transformers", "accelerate", "av", "timm", "einops", "scipy"]
missing_pkgs = []
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"[*] Post-restart auto-fix: Missing packages detected ({', '.join(missing_pkgs)})...\n[*] Installing required dependencies...")
    !pip install -q av diffusers transformers accelerate fvcore timm omegaconf einops scikit-image huggingface_hub onnxruntime-gpu opencv-python tqdm pillow psutil pycocotools pyyaml cloudpickle scipy
    print("✅ LEFFA dependencies successfully installed!")

# 2. Pre-flight check: ensure LEFFA repository is present and sys.path is configured
LEFFA_ROOT = "/content/Leffa"
if not os.path.exists(os.path.join(LEFFA_ROOT, "leffa", "transform.py")):
    print("[*] Pre-flight auto-fix: Cloning official LEFFA repository...")
    !rm -rf /content/Leffa
    !git clone https://github.com/franciszzj/Leffa.git /content/Leffa

for p in [LEFFA_ROOT, os.path.join(LEFFA_ROOT, "3rdparty")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# 3. Clean write of leffa/inference.py with official LeffaPipeline architecture and safe generator fallback
inference_py = os.path.join(LEFFA_ROOT, "leffa", "inference.py")
if os.path.exists(os.path.dirname(inference_py)):
    with open(inference_py, "w", encoding="utf-8") as f:
        f.write("""from typing import Any, Dict, Optional
import numpy as np
import torch
import torch.nn as nn
from leffa.pipeline import LeffaPipeline

class LeffaInference(object):
    def __init__(self, model: nn.Module) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device)
        self.model.eval()
        self.pipe = LeffaPipeline(model=self.model, device=self.device)

    def to_gpu(self, data: Dict[str, Any]) -> Dict[str, Any]:
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                data[k] = v.to(self.device)
        return data

    def __call__(self, data: Dict[str, Any], **kwargs) -> Dict[str, Any]:
        data = self.to_gpu(data)
        ref_acceleration = kwargs.get("ref_acceleration", False)
        num_inference_steps = kwargs.get("num_inference_steps", 50)
        guidance_scale = kwargs.get("guidance_scale", 2.5)
        seed = kwargs.get("seed", 42)
        repaint = kwargs.get("repaint", False)
        try:
            generator = torch.Generator(self.pipe.device).manual_seed(seed)
        except Exception:
            generator = torch.Generator("cpu").manual_seed(seed)
        images = self.pipe(
            src_image=data["src_image"],
            ref_image=data["ref_image"],
            mask=data["mask"],
            densepose=data["densepose"],
            ref_acceleration=ref_acceleration,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
            repaint=repaint,
        )[0]
        outputs = {}
        outputs["src_image"] = (data["src_image"] + 1.0) / 2.0
        outputs["ref_image"] = (data["ref_image"] + 1.0) / 2.0
        outputs["generated_image"] = images
        return outputs
""")

# 4. Directory and Checkpoint Auto-Healer
WORKSPACE_DIR = globals().get("WORKSPACE_DIR", "/content/virtualfit_workspace")
INPUT_DIR = globals().get("INPUT_DIR", os.path.join(WORKSPACE_DIR, "input"))
OUTPUT_DIR = globals().get("OUTPUT_DIR", os.path.join(WORKSPACE_DIR, "output"))
CKPTS_DIR = globals().get("CKPTS_DIR", "/content/ckpts")

required_vt = os.path.join(CKPTS_DIR, "virtual_tryon.pth")
if not os.path.exists(required_vt):
    print(f"[*] Pre-flight auto-fix: Model checkpoints missing in {CKPTS_DIR} after restart.\n[*] Downloading targeted LEFFA weights (~8 GB)...")
    from huggingface_hub import snapshot_download
    os.makedirs(CKPTS_DIR, exist_ok=True)
    snapshot_download(
        repo_id="franciszzj/Leffa",
        local_dir=CKPTS_DIR,
        allow_patterns=[
            "stable-diffusion-inpainting/**",
            "virtual_tryon.pth",
            "virtual_tryon_dc.pth",
            "densepose/**",
            "humanparsing/**",
            "openpose/**",
            "schp/**"
        ],
        ignore_patterns=[
            "stable-diffusion-xl-1.0-inpainting-0.1/**",
            "pose_transfer.pth",
            "examples/**"
        ]
    )
    print("✅ Model checkpoints successfully restored!")

# Restore task metadata if Step 4 was not executed in current session
if "metadata" not in globals():
    meta_path = os.path.join(INPUT_DIR, "metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
    else:
        raise FileNotFoundError("❌ No VirtualFit task package found in workspace. Please run Step 4 first to upload your ZIP.")

raw_cat = metadata.get("category", "tops")
garment_type = metadata.get("leffa_garment_type")
if not garment_type:
    cat_map = {"tops": "upper_body", "bottoms": "lower_body", "one-pieces": "dresses"}
    garment_type = cat_map.get(raw_cat, "upper_body")

model_type = metadata.get("leffa_model_type", "dress_code" if garment_type == "dresses" else "viton_hd")
weights_filename = "virtual_tryon_dc.pth" if model_type == "dress_code" else "virtual_tryon.pth"

# Core LEFFA imports
from leffa.transform import LeffaTransform
from leffa.model import LeffaModel
from leffa.inference import LeffaInference
from leffa_utils.utils import resize_and_center, get_agnostic_mask_hd, get_agnostic_mask_dc
from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose

has_densepose_module = False
try:
    from leffa_utils.densepose_predictor import DensePosePredictor
    has_densepose_module = True
except Exception as dp_imp_err:
    print(f"[Notice] DensePose import notice: {dp_imp_err}")
    has_densepose_module = False

def log_mem(tag=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        alloc = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        print(f"[Memory {tag}] VRAM Allocated: {alloc:.2f} GB | Reserved: {reserved:.2f} GB")

def load_image_clean(image_path: str) -> Image.Image:
    img = Image.open(image_path)
    if img.mode in ("RGBA", "LA") or ("transparency" in img.info):
        alpha = img.convert("RGBA")
        bg = Image.new("RGBA", alpha.size, (255, 255, 255, 255))
        bg.paste(alpha, mask=alpha.split()[3])
        return bg.convert("RGB")
    return img.convert("RGB")

def find_input_file(directory: str, prefix: str) -> str:
    for ext in [".png", ".jpg", ".jpeg", ".webp"]:
        candidate = os.path.join(directory, f"{prefix}{ext}")
        if os.path.exists(candidate):
            return candidate
    for f in os.listdir(directory):
        if f.lower().startswith(prefix.lower()) and any(f.lower().endswith(ext) for ext in [".png", ".jpg", ".jpeg", ".webp"]):
            return os.path.join(directory, f)
    raise FileNotFoundError(f"Could not find '{prefix}' image in {directory}")

start_time = time.time()
log_mem("Pre-Execution Baseline")

person_path = find_input_file(INPUT_DIR, "person")
garment_path = find_input_file(INPUT_DIR, "garment")
output_png_path = os.path.join(OUTPUT_DIR, "output.png")

# 1. Load and resize to exact target resolution: 768 x 1024
src_image = load_image_clean(person_path)
src_image = resize_and_center(src_image, 768, 1024)

ref_image = load_image_clean(garment_path)
ref_image = resize_and_center(ref_image, 768, 1024)

src_image_array = np.array(src_image)

# Save debugging verification files
src_image.save(os.path.join(WORKSPACE_DIR, "debug_person_input.png"))
ref_image.save(os.path.join(WORKSPACE_DIR, "debug_garment_input.png"))
with open(os.path.join(WORKSPACE_DIR, "debug_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# 2. Human Parsing & OpenPose (Fast Preprocessing)
print("\n[*] Running Human Parsing & Keypoint Detection...")
parsing = Parsing(
    atr_path=os.path.join(CKPTS_DIR, "humanparsing", "parsing_atr.onnx"),
    lip_path=os.path.join(CKPTS_DIR, "humanparsing", "parsing_lip.onnx"),
)
openpose = OpenPose(
    body_model_path=os.path.join(CKPTS_DIR, "openpose", "body_pose_model.pth"),
)

model_parse, _ = parsing(src_image.resize((384, 512)))
keypoints = openpose(src_image.resize((384, 512)))

# 3. Compute Agnostic Mask
if model_type == "dress_code":
    mask = get_agnostic_mask_dc(model_parse, keypoints, garment_type)
else:
    mask = get_agnostic_mask_hd(model_parse, keypoints, garment_type)
mask = mask.resize((768, 1024))

# 4. DensePose Geometry Conditioning with Fault-Tolerance
print("[*] Computing DensePose Surface Coordinates...")
densepose = None
if has_densepose_module:
    try:
        densepose_predictor = DensePosePredictor(
            config_path=os.path.join(CKPTS_DIR, "densepose", "densepose_rcnn_R_50_FPN_s1x.yaml"),
            weights_path=os.path.join(CKPTS_DIR, "densepose", "model_final_162be9.pkl"),
        )
        if model_type == "dress_code":
            src_image_iuv = densepose_predictor.predict_iuv(src_image_array)
            src_seg = src_image_iuv[:, :, 0:1]
            src_seg = np.concatenate([src_seg] * 3, axis=-1)
            densepose = Image.fromarray(src_seg)
        else:
            src_seg = densepose_predictor.predict_seg(src_image_array)[:, :, ::-1]
            densepose = Image.fromarray(src_seg)
        del densepose_predictor
        log_mem("Post-DensePose Cleanup")
    except Exception as dp_err:
        print(f"[Warning] DensePose notice ({dp_err}). Generating fallback geometry map...")
        densepose = None

if densepose is None:
    parse_np = np.array(model_parse.resize((768, 1024), Image.NEAREST))
    parse_rgb = np.stack([parse_np * 12, parse_np * 8, parse_np * 15], axis=-1).astype(np.uint8)
    densepose = Image.fromarray(parse_rgb)
    print("    [Fallback] Body geometry conditioning surface map generated successfully.")

# Evict Preprocessing Models from GPU VRAM
print("[*] Evicting Preprocessors from GPU memory...")
del parsing
del openpose
del model_parse
del keypoints
log_mem("Post-Preprocessing Cleanup (Preprocessors Freed)")

# 5. Memory-Optimized LeffaModel Loading (mmap=True & float16 streaming)
t_model_start = time.time()
weights_path = os.path.join(CKPTS_DIR, weights_filename)
sd_path = os.path.join(CKPTS_DIR, "stable-diffusion-inpainting")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = "float16" if device == "cuda" else "float32"

print(f"\n[*] Streaming {weights_filename} in {dtype} (Zero-Duplicate RAM Loader)...")
leffa_model = LeffaModel(
    pretrained_model_name_or_path=sd_path,
    pretrained_model="",  # Skip internal FP32 loading
    dtype=dtype,
)
if device == "cuda":
    leffa_model.half()

# Stream weights with mmap=True
state_dict = torch.load(weights_path, map_location="cpu", mmap=True)
for k in list(state_dict.keys()):
    if torch.is_floating_point(state_dict[k]):
        state_dict[k] = state_dict[k].half() if device == "cuda" else state_dict[k].float()

leffa_model.load_state_dict(state_dict)
del state_dict
gc.collect()

if device == "cuda":
    leffa_model.to("cuda")
    torch.cuda.empty_cache()

model_load_time = time.time() - t_model_start
print(f"[*] Model loading completed in {model_load_time:.2f} seconds on {device}.")
log_mem("Post-Model Load")

# 6. Transform Input Data & Execute Flow-Field Attention Diffusion
leffa_inference = LeffaInference(model=leffa_model)

print("\n[*] Transforming input tensors with LeffaTransform...")
transform = LeffaTransform()
data = {
    "src_image": [src_image],
    "ref_image": [ref_image],
    "mask": [mask],
    "densepose": [densepose],
}
data = transform(data)

print("\n[*] Running Neural Diffusion (30 timesteps, CFG 2.5, ref_acceleration=True)...")
t_infer_start = time.time()
with torch.inference_mode():
    output = leffa_inference(
        data,
        ref_acceleration=True,
        num_inference_steps=30,
        guidance_scale=2.5,
        seed=42,
        repaint=False,
    )

infer_time = time.time() - t_infer_start
output_image = output["generated_image"][0]
output_image.save(output_png_path)
output_image.save(os.path.join(WORKSPACE_DIR, "debug_leffa_output.png"))

peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
print(f"\n🎉 GENUINE LEFFA VIRTUAL TRY-ON COMPLETED SUCCESSFULLY!")
print(f"   • Output Resolution:  {output_image.size[0]}x{output_image.size[1]} px")
print(f"   • Model Loading Time: {model_load_time:.2f} s")
print(f"   • Inference Time:     {infer_time:.2f} s")
print(f"   • Peak VRAM Used:     {peak_vram:.2f} GB")
print(f"   • Saved to:           {output_png_path}")

# 7. Cleanup & VRAM Release
del output
del data
del transform
del leffa_inference
del leffa_model
log_mem("Final Post-Inference Release")


In [ ]:
# Step 6: Preview Results & Download Output Result ZIP
import matplotlib.pyplot as plt
from datetime import datetime
from google.colab import files

result_img = Image.open(output_png_path)

# Display 3-panel side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
axes[0].imshow(src_image)
axes[0].set_title("Original Person", fontsize=13, pad=8)
axes[0].axis("off")

axes[1].imshow(ref_image)
axes[1].set_title(f"Selected Garment ({garment_type})", fontsize=13, pad=8)
axes[1].axis("off")

axes[2].imshow(result_img)
axes[2].set_title("Genuine LEFFA AI (CVPR 2025) Result", fontsize=13, pad=8, color="#4A337E", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

# Package result ZIP for VirtualFit AI
generation_id = metadata.get("generation_id", "vfit_output")
result_zip_name = f"virtualfit_result_{generation_id}.zip"
result_zip_path = os.path.join(WORKSPACE_DIR, result_zip_name)

metadata["status"] = "success"
metadata["completed_at"] = datetime.utcnow().isoformat()
metadata["output_dimensions"] = [result_img.size[0], result_img.size[1]]
metadata["engine"] = "LEFFA (Learning Flow Fields in Attention, CVPR 2025)"
metadata["engine_id"] = "leffa"
metadata["inference_metrics"] = {
    "model_load_time_sec": round(model_load_time, 2),
    "inference_time_sec": round(infer_time, 2),
    "peak_vram_gb": round(peak_vram, 2),
}

with zipfile.ZipFile(result_zip_path, 'w', zipfile.ZIP_DEFLATED) as zip_out:
    zip_out.write(output_png_path, arcname="output.png")
    zip_out.writestr("metadata.json", json.dumps(metadata, indent=2))

print(f"\n✅ Successfully packaged result into: {result_zip_name}")
print("📥 Initiating automatic browser download...")
try:
    files.download(result_zip_path)
except Exception as e:
    print(f"[Notice] Result ZIP saved to: {result_zip_path} (can also download directly from Colab files pane).")

print("\n👉 Next step: Go to the VirtualFit AI website, upload this ZIP, and explore the before-and-after slider!")
